# Phase 2: Data Preprocessing & HI Construction (Proposed Method - Cross Condition)

This notebook implements the Health Index (HI) construction pipeline based on the proposed method by Nemani et al. (2022) with a Cross-Condition split on the XJTU-SY dataset.

**Pipeline Steps:**
1.  **Data Acquisition**: Load raw vibration data for all 15 bearings across 3 operating conditions.
2.  **Feature Extraction**: Convert Acceleration to Velocity and extract 124 features per minute.
3.  **Data Splitting**: Global split of 12 Bearings for Training and 3 Bearings for Validation (`Bearing1_5`, `Bearing2_5`, `Bearing3_5`).
4.  **Feature Selection**: Global evaluation of Meta-Probability (Spearman and Modified Monotonicity >40th percentile) calculated purely on the 12 Training Bearings.
5.  **HI Construction**: LHS Optimization (2000 combinations) across the 12 Training Bearings to find the optimal fusion weights.
6.  **Inference & Evaluation**: Apply weights to the 3 Validation Bearings and compute final metrics individually.
7.  **Diagnostic Logging**: Plot HI curves and save selected features and correlations to CSV.


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.integrate import cumulative_trapezoid
from scipy.fft import fft
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.stats import qmc
from typing import List, Dict, Tuple, Callable
import warnings
warnings.filterwarnings('ignore')

# Add src path
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), 'src')))


In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================
RAW_DATA_PATH = r"D:\Proyek Dosen\Riset Bearing\XJTU-SY_Bearing_Datasets"
OUTPUT_HI_PATH = r"D:\Proyek Dosen\Riset Bearing\Notebook-Github\3rd Research_Cross-Domain Generalization RUL Bearing with XAI\Processed_HI"
CSV_LOG_PATH = os.path.join(OUTPUT_HI_PATH, "Feature_Logs")

os.makedirs(OUTPUT_HI_PATH, exist_ok=True)
os.makedirs(CSV_LOG_PATH, exist_ok=True)

SAMPLING_FREQ = 25600
ROWS_PER_FILE = 32768
TARGET_CONDITIONS = ['35Hz12kN', '37.5Hz11kN', '40Hz10kN']
TEST_BEARINGS = ['Bearing1_5', 'Bearing2_5', 'Bearing3_5']


In [ ]:
def acc_to_vel(acc_signal: np.ndarray, fs: float) -> np.ndarray:
    """Integrates acceleration to velocity and detrends the result."""
    dt = 1.0 / fs
    vel = cumulative_trapezoid(acc_signal, dx=dt, initial=0.0)
    vel_detrended = vel - np.mean(vel)
    return vel_detrended * 1000 # Convert to mm/s


In [ ]:
from ConstructHI.FeatureExtractionAndSelection import FeatureExtractionAndSelection

class XJTUFeatureExtractor(FeatureExtractionAndSelection):
    def __init__(self, data_directory: str):
        super().__init__(data_directory)
        self.sampling_frequency_hz = 25600
    
    def process_bearing_data(self, bearing_path: str, shaft_freq: float) -> np.ndarray:
        csv_files = sorted(glob.glob(os.path.join(bearing_path, "*.csv")))
        features_list = []
        
        for file in csv_files:
            df = pd.read_csv(file)
            if 'Horizontal_vibration_signals' in df.columns:
                h_acc = df['Horizontal_vibration_signals'].values
                v_acc = df['Vertical_vibration_signals'].values
            else:
                h_acc = df.iloc[:, 0].values
                v_acc = df.iloc[:, 1].values
            
            # Truncate or pad to exactly 32768
            if len(h_acc) > 32768: h_acc = h_acc[:32768]
            if len(v_acc) > 32768: v_acc = v_acc[:32768]
            
            h_vel = acc_to_vel(h_acc, self.sampling_frequency_hz)
            v_vel = acc_to_vel(v_acc, self.sampling_frequency_hz)
            
            signals = [h_acc, h_vel, v_acc, v_vel]
            row_features = []
            
            for sig in signals:
                t_feat = self._extract_time_features(sig)
                f_feat = self._extract_frequency_features(sig, shaft_freq)
                row_features.extend(t_feat)
                row_features.extend(f_feat)
                
            features_list.append(row_features)
            
        return np.array(features_list)


In [ ]:
from sklearn.preprocessing import MinMaxScaler
from ConstructHI.HealthIndexConstruction import HealthIndexConstruction

def run_cross_condition_pipeline():
    print(f"\n{'='*50}\nStarting Global Cross-Condition Pipeline\n{'='*50}")
    
    extractor = XJTUFeatureExtractor(RAW_DATA_PATH)
    hi_construct = HealthIndexConstruction(num_samples_lhs=2000)
    
    all_features = {}
    train_bearings = []
    val_bearings = []
    
    # 1. Acquire and Extract Features for all conditions and bearings
    for condition in TARGET_CONDITIONS:
        cond_path = os.path.join(RAW_DATA_PATH, condition)
        if not os.path.exists(cond_path):
            print(f"Skipping {condition}, folder not found.")
            continue
            
        shaft_freq = 35.0
        if '37.5Hz' in condition: shaft_freq = 37.5
        elif '40Hz' in condition: shaft_freq = 40.0
        
        bearings = sorted([f for f in os.listdir(cond_path) if os.path.isdir(os.path.join(cond_path, f))])
        
        for bearing in bearings:
            print(f"Extracting features for {condition} / {bearing}...")
            feats = extractor.process_bearing_data(os.path.join(cond_path, bearing), shaft_freq)
            uid = f"{condition}_{bearing}"
            all_features[uid] = feats
            
            if bearing in TEST_BEARINGS:
                val_bearings.append(uid)
            else:
                train_bearings.append(uid)
                
    print(f"\nTrain Bearings ({len(train_bearings)}): {train_bearings}")
    print(f"Validation Bearings ({len(val_bearings)}): {val_bearings}")
    
    if len(train_bearings) == 0:
        print("No training data found. Exiting.")
        return

    # 2. Extract Data Lists
    train_data_list = [all_features[uid] for uid in train_bearings]
    train_fpts = [0] * len(train_bearings)
    
    # 3. Global Scaling
    scaler = MinMaxScaler()
    concat_train = np.vstack(train_data_list)
    scaler.fit(concat_train)
    
    scaled_train_data = [scaler.transform(td) for td in train_data_list]
    
    scaled_val_data_dict = {}
    for uid in val_bearings:
        scaled_val_data_dict[uid] = scaler.transform(all_features[uid])
        
    # 4. Global Feature Selection using Train Set Population
    num_features = scaled_train_data[0].shape[1]
    spearman_meta = []
    mod_mon_meta = []
    
    print("\nEvaluating Meta-Probabilities for Feature Selection...")
    for i in range(num_features):
        feat_sp = []
        feat_mm = []
        for td in scaled_train_data:
            deg_phase = td[:, i]
            sp, _ = stats.spearmanr(deg_phase, np.arange(len(deg_phase)))
            mm = extractor.calculate_modified_monotonicity(deg_phase, sigma=0.01)
            feat_sp.append(abs(sp))
            feat_mm.append(mm)
        
        sp_prob, _, _ = extractor.calculate_meta_probability(feat_sp, (0.4, 1.0))
        mm_prob, _, _ = extractor.calculate_meta_probability(feat_mm, (0.4, 1.0))
        spearman_meta.append(sp_prob)
        mod_mon_meta.append(mm_prob)
        
    sp_thresh = np.percentile(spearman_meta, 40)
    mm_thresh = np.percentile(mod_mon_meta, 40)
    
    selected_idx = np.where((np.array(spearman_meta) > sp_thresh) & (np.array(mod_mon_meta) > mm_thresh))[0]
    print(f"Selected {len(selected_idx)} features out of {num_features} (Indices: {selected_idx[:10]}...)")
    
    filtered_train = [td[:, selected_idx] for td in scaled_train_data]
    filtered_val_dict = {uid: data[:, selected_idx] for uid, data in scaled_val_data_dict.items()}
    
    # 5. HI Construction Optimization (LHS on Train Set)
    print("\nOptimizing global weights via LHS (2000 combinations)...")
    best_weights, best_metrics = hi_construct.generate_optimal_hi(
        filtered_train, train_fpts,
        extractor.calculate_meta_probability,
        lambda deg, sig: extractor.calculate_modified_monotonicity(deg, sig)
    )
    
    print(f"Optimal Weights Training Score: {best_metrics['combined_score']:.4f}")
    
    # Save Feature Logs globally
    csv_df = pd.DataFrame({
        'Selected_Feature_Index': selected_idx,
        'Spearman_MetaProb': np.array(spearman_meta)[selected_idx],
        'ModMon_MetaProb': np.array(mod_mon_meta)[selected_idx],
        'Final_Weight': best_weights
    })
    csv_df.to_csv(os.path.join(CSV_LOG_PATH, f"Global_Cross_Condition_Features.csv"), index=False)
    
    # 6. Inference and Evaluation on Validation Bearings
    print(f"\n{'='*50}\nInference on Validation Set\n{'='*50}")
    
    for uid in val_bearings:
        hi_val = np.dot(filtered_val_dict[uid], best_weights)
        hi_val = (hi_val - hi_val.min()) / (hi_val.max() - hi_val.min() + 1e-8) # Normalize HI to 0-1
        
        metrics = hi_construct.get_metrics(hi_val, 0, lambda deg, sig: extractor.calculate_modified_monotonicity(deg, sig))
        
        print(f"\n--- Validation Metrics: {uid} ---")
        for k, v in metrics.items():
            print(f"  - {k}: {v:.4f}")
            
        plt.figure(figsize=(10, 4))
        plt.plot(hi_val, label='Constructed HI', color='green')
        plt.title(f"Health Index - {uid} (Cross-Condition Validation)")
        plt.xlabel("Time (Minutes)")
        plt.ylabel("Normalized Health Index")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_HI_PATH, f"Validation_{uid}_HI.png"))
        plt.show()

# EXECUTE 
if __name__ == "__main__":
    run_cross_condition_pipeline()
